In [1]:
%matplotlib qt
%reload_ext autoreload
%autoreload 2
import lf_spec
import plotter
import sgd
import classifier
import numpy as np
import os
from tqdm import tqdm

lf_spec.lf_connect()
sgd.sgd_init()

LightField bridge already running.
Scanning for instruments...
Getting ready...
Ready for use!


## Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `foldername` | str | Base name of the folder where data will be saved. |
| `scan_type` | str | Subfolder category. Options: `"coarse"` or `"fine"`. |
| `xdim`, `ydim` | float | Scan area width/height in um. Set to `0` for single-point. |
| `dx`, `dy` | float | Step size in um. |
| `center` | tuple | `(x, y)` centre of scan in um. |
| `grating` | int | `150` (150 g/mm) or `600` (600 g/mm). |
| `exposure_s` | float | Integration time per point in seconds. |
| `center_wl` | int | Spectrometer centre wavelength in nm. |

In [3]:
foldername  = '20260519-PLSPC-example'
scan_type   = 'coarse'
data_folder = 'data'

xdim, ydim  = 1.0, 1.0
dx,   dy    = 0.5, 0.5
center      = (0.0, 0.0)
grating     = 150
exposure_s  = 1.0
center_wl   = 700

lf_spec.lf_setup(exposure_s=exposure_s, center_wavelength=center_wl, grating=grating)

folder_path = os.path.join(data_folder, foldername, scan_type)
os.makedirs(folder_path, exist_ok=True)

wl  = lf_spec.lf_get_wavelengths()
num = len(wl)
np.save(os.path.join(folder_path, 'wl.npy'), wl)
print(f'WL: {wl[0]:.1f}-{wl[-1]:.1f} nm  ({num} points)')

xs = np.arange(-xdim/2 + center[0], xdim/2 + dx + center[0], dx)
ys = np.arange(-ydim/2 + center[1], ydim/2 + dy + center[1], dy)
output = np.zeros((len(ys), len(xs), num))

sgd.sgd_on()
with tqdm(total=len(ys)*len(xs), desc='Scanning') as pbar:
    for iy, y in enumerate(ys):
        for ix, x in enumerate(xs):
            sgd.set_position(x, y, silent=True)
            intensity, wl_acq = lf_spec.lf_acquire()
            intensity = np.array(intensity)
            if len(intensity) != num:
                intensity = np.resize(intensity, num)
            output[iy, ix, :] = intensity
            wl = wl_acq
            pbar.set_description(f'x={x:.2f}, y={y:.2f}')
            pbar.update(1)
sgd.sgd_off()

np.save(os.path.join(folder_path, 'out.npy'), output)
np.save(os.path.join(folder_path, 'xs.npy'), xs)
np.save(os.path.join(folder_path, 'ys.npy'), ys)
np.save(os.path.join(folder_path, 'wl.npy'), wl)

classifier.classify_all(foldername, scan_type, data_folder=data_folder)
print('Done.')

WL: 415.6-980.1 nm  (1340 points)
Sgd on!


x=0.50, y=0.50: 100%|██████████| 9/9 [00:21<00:00,  2.43s/it]  


Sgd off!
Found 0 distinct emitters.
Done.


In [4]:
plotter.plot_heatmap_manual(foldername, scan_type)